<a href="https://colab.research.google.com/github/jasonkwh/mario-snes-cnn-ppo/blob/main/trained_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%pip install -U gymnasium stable-retro stable-baselines3 opencv-python

  Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (38 kB)
Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl (526.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.13.0 which is incompatible.


In [ ]:
import sys
import subprocess
import gc
import torch
import gymnasium as gym
import numpy as np
import stable_retro
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.monitor import Monitor

class RewardWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.prev_x = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_x = info.get("x", 0)
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        current_x = info.get("x", 0)
        x_reward = current_x - self.prev_x
        self.prev_x = current_x

        custom_reward = x_reward * 0.1
        if terminated:
            custom_reward -= 15.0  # Death penalty

        return obs, custom_reward, terminated, truncated, info


def make_env(
    game_name,
    state_name,
    record_video=False,
    video_folder="./mario_videos",
    monitor_filename="./monitor.csv"):
    env = stable_retro.make(
        game=game_name,
        state=state_name,
        use_restricted_actions=stable_retro.Actions.DISCRETE,
        render_mode="rgb_array",
    )

    env = RewardWrapper(env)
    env = MaxAndSkipEnv(env, skip=4)
    env = gym.wrappers.ResizeObservation(env, shape=(84, 96))
    env = gym.wrappers.GrayscaleObservation(env, keep_dim=True)

    # Add RecordVideo as the final wrapper
    if record_video:
        env = gym.wrappers.RecordVideo(
            env,
            video_folder=video_folder,
            episode_trigger=lambda episode_id: episode_id % 100 == 0 # Records every 100th episode
        )

    env = Monitor(env, filename=monitor_filename)

    return env


if __name__ == "__main__":
    game_name = "SuperMarioWorld-Snes-v0"
    state_name = "YoshiIsland1"
    model_name = "mario_ppo"
    record_video = True
    checkpoint_dir = "./mario_checkpoints/"
    video_dir = "./mario_videos/"
    monitor_filename = "./monitor.csv"
    # 1. Clean up old environment instances stuck in memory from previous Colab runs
    try:
        if 'env' in locals():
            env.close()
            del env
    except Exception:
        pass

    gc.collect()

    # 2. Run ROM import ONCE before training starts
    result = subprocess.run(
        [sys.executable, "-m", "stable_retro.import", "."],
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    # 3. Vectorize using Lambda factory, Stack, and Transpose for PyTorch CNN
    env = DummyVecEnv([lambda: make_env(game_name, state_name, record_video, video_dir, monitor_filename)])
    env = VecFrameStack(env, n_stack=4, channels_order="last")
    env = VecTransposeImage(env)

    # Automatically check CUDA availability
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 4. Initialize PPO Agent
    model = PPO(
        policy="CnnPolicy",
        env=env,
        device=device,
        verbose=1,
        learning_rate=0.0001,
        n_steps=2048,
        batch_size=64,
        ent_coef=0.01,
    )

    # 5. Train Policy
    print(f"Training {game_name} Agent on {device.upper()}...")
    model.learn(
        total_timesteps=1000000,
        callback=CheckpointCallback(
            save_freq=50000,
            save_path=checkpoint_dir,
            name_prefix=model_name,
            verbose=2
        )
    )

    # 6. Save Weights
    model.save(model_name)
    print(f"Model saved successfully as '{model_name}.zip'")